# Kinyarwanda pseudo-labeling on Colab GPU

Runs the same transcribe + KenLM-correction pipeline as the local CPU version, but on a free Colab GPU (15-25x faster).

## One-time setup (do this before running any cells)
1. In Google Drive, create a folder: `KinyarwandaASR/`
2. Inside it, create three subfolders: `assets/`, `audio_in/`, `labels_out/`
3. Upload these two files from your machine into `KinyarwandaASR/assets/`:
   - `kenlm_model.binary`
   - `lm_corpus.txt`
   - `normalize.py`
   - `lm_rescore.py`
4. Each session: upload a manageable batch of audio files (a few hours' worth) into `KinyarwandaASR/audio_in/`.
5. Run all cells below (Runtime -> Run all). Make sure Runtime -> Change runtime type -> GPU (T4) is selected.
6. Outputs land in `KinyarwandaASR/labels_out/` as one JSON per audio file -- already-processed files are skipped automatically, so it's safe to re-run across multiple sessions as you add more audio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/KinyarwandaASR'
ASSETS_DIR = f'{PROJECT_DIR}/assets'
AUDIO_DIR = f'{PROJECT_DIR}/audio_in'
OUTPUT_DIR = f'{PROJECT_DIR}/labels_out'

import os
for d in (PROJECT_DIR, ASSETS_DIR, AUDIO_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

missing = [f for f in ('kenlm_model.binary', 'lm_corpus.txt', 'normalize.py', 'lm_rescore.py')
           if not os.path.exists(f'{ASSETS_DIR}/{f}')]
if missing:
    print('Missing from assets/, upload these before continuing:', missing)
else:
    print('All assets present.')

In [ ]:
!pip install -q faster-whisper kenlm

In [ ]:
# Download the fine-tuned Kinyarwanda Whisper model directly from Hugging Face (fast on Colab's network)
import os
MODEL_DIR = '/content/whisper-kinyarwanda-ct2'
os.makedirs(MODEL_DIR, exist_ok=True)
REPO = 'leophill/whisper-large-v3-turbo-sw-kinyarwanda-ct2'
BASE_URL = f'https://huggingface.co/{REPO}/resolve/main'
for f in ['config.json', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json', 'model.bin']:
    dest = f'{MODEL_DIR}/{f}'
    if not os.path.exists(dest):
        !curl -L -C - --retry 50 --retry-delay 5 --retry-all-errors -o "{dest}" "{BASE_URL}/{f}"
print('Model ready at', MODEL_DIR)

In [ ]:
# Make the uploaded normalize.py / lm_rescore.py importable, pointed at the Drive assets
import sys, shutil
sys.path.insert(0, ASSETS_DIR)

# lm_rescore.py / normalize.py expect their data files next to themselves (Path(__file__).parent);
# since they already live in ASSETS_DIR alongside kenlm_model.binary and lm_corpus.txt, no changes needed.

from normalize import normalize_hypothesis
from lm_rescore import correct_words
print('Loaded normalize_hypothesis + correct_words OK')

In [ ]:
import json, time
from pathlib import Path
from faster_whisper import WhisperModel

AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg', '.webm')

def output_path_for(audio_path: Path) -> Path:
    safe_stem = ''.join(c if c.isalnum() else '_' for c in audio_path.stem)[:80]
    return Path(OUTPUT_DIR) / f'{safe_stem}.json'

print('Loading model on GPU...')
model = WhisperModel(MODEL_DIR, device='cuda', compute_type='float16')

all_files = sorted(p for p in Path(AUDIO_DIR).iterdir() if p.suffix.lower() in AUDIO_EXTS)
todo = [p for p in all_files if not output_path_for(p).exists()]
print(f'{len(all_files)} audio files in audio_in/, {len(todo)} not yet labeled.')

for i, audio_path in enumerate(todo, 1):
    out_path = output_path_for(audio_path)
    print(f'[{i}/{len(todo)}] {audio_path.name}')
    start = time.time()
    try:
        segments, info = model.transcribe(
            str(audio_path),
            language='sw',
            task='transcribe',
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=300),
            condition_on_previous_text=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            compression_ratio_threshold=2.4,
        )
        seg_records, raw_parts = [], []
        for seg in segments:
            text = seg.text.strip()
            raw_parts.append(text)
            seg_records.append({
                'start': seg.start, 'end': seg.end, 'text': text,
                'avg_logprob': seg.avg_logprob,
                'no_speech_prob': seg.no_speech_prob,
                'compression_ratio': seg.compression_ratio,
            })
        raw_text = ' '.join(raw_parts)
        corrected_text = correct_words(normalize_hypothesis(raw_text))
        elapsed = time.time() - start
        avg_lp = [s['avg_logprob'] for s in seg_records]
        nsp = [s['no_speech_prob'] for s in seg_records]
        record = {
            'source_path': str(audio_path),
            'duration_s': info.duration,
            'transcribe_time_s': elapsed,
            'detected_language': info.language,
            'language_probability': info.language_probability,
            'raw_text': raw_text,
            'corrected_text': corrected_text,
            'segments': seg_records,
            'mean_avg_logprob': sum(avg_lp) / len(avg_lp) if avg_lp else None,
            'mean_no_speech_prob': sum(nsp) / len(nsp) if nsp else None,
            'num_segments': len(seg_records),
        }
        out_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'  duration={record["duration_s"]:.1f}s transcribe_time={record["transcribe_time_s"]:.1f}s '
              f'mean_avg_logprob={record["mean_avg_logprob"]:.3f}')
    except Exception as e:
        print(f'  ERROR: {e}')

print('Done for this session. Outputs in', OUTPUT_DIR)